# Colonial Visual Grammar Analysis

**Hypothesis:** SD doesn't just produce biased *associations* — it reproduces **colonial visual grammar** as a generative process. When prompted with "Arab person" vs "a person", the model's latent trajectory makes systematic spatial moves that can be isolated by averaging out noise across seeds.

**Method:**
1. Generate N variations per prompt category (different seeds, same prompt) on the **remote GPU**
2. Average latent trajectories → noise cancels, systematic patterns emerge
3. Compute flow fields (velocity at each step) and their differences
4. Track *when* during denoising the category-specific patterns lock in
5. Find *where* in the 64×64 latent grid the differences concentrate
6. (Optional) Project onto colonial archive PC axes if historical images are available

In [ ]:
import sys
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from io import BytesIO
import json

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from client.interface import SlopClient, InferenceResult
from client.config import registry
from client.analysis.colonial import (
    ColonialGrammarAnalysis,
    # Batch-aware statistics
    batch_divergence,
    batch_flow_difference,
    batch_channel_difference,
    batch_spatial_heatmap,
    batch_inter_category_pca,
    BatchDivergenceResult,
    BatchFlowResult,
    BatchChannelResult,
    BatchSpatialResult,
    # Divergence rate analysis
    batch_divergence_rate,
    compare_divergence_rates,
    DivergenceRateResult,
    # Spatial motif extraction
    extract_motif_direction,
    measure_motif_activation,
    extract_motif_components,
    MotifDirection,
    MotifExtractionResult,
    # Visualization helpers
    average_trajectories,
    compute_flow_field,
    average_flow_field,
    flow_difference,
    flow_magnitude,
    spatial_difference_heatmap,
    temporal_heatmap_sequence,
    drift_score,
)

print("Imports OK — batch stats, divergence rates, motif extraction ready")

## Configuration

In [ ]:
# ── Server ──────────────────────────────────────────────────────────
SERVER_NAME = "vast-auto-test"
MODEL = "runwayml/stable-diffusion-v1-5"

# ── Generation params ───────────────────────────────────────────────
N_VARIATIONS = 10          # 10 seeds per category (increase for cleaner signal)
NUM_STEPS = 50             # denoising steps
GUIDANCE_SCALE = 7.5
SEED_START = 0

# ── Prompt categories ───────────────────────────────────────────────
CATEGORIES = {
    "neutral":     "a person in daily life",
    "arab":        "an Arab person in daily life",
    "palestinian": "a Palestinian person in daily life",
}

# ── Output ──────────────────────────────────────────────────────────
OUTPUT_DIR = project_root / "results" / "colonial_grammar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = registry.get(SERVER_NAME)
if not cfg:
    raise ValueError(f"Server {SERVER_NAME} not found in registry. Run deploy first.")

print(f"Server:     {SERVER_NAME}")
print(f"Model:      {MODEL}")
print(f"Categories: {list(CATEGORIES.keys())}")
print(f"Seeds/cat:  {N_VARIATIONS}")
print(f"Steps:      {NUM_STEPS}")
print(f"Est. time:  ~{N_VARIATIONS * len(CATEGORIES) * 5 / 60:.0f} min on RTX 3090")

---
## 1. Batch Generation

Generate `N_VARIATIONS` images per category. Each gets a sequential seed so results are reproducible. We capture the full latent trajectory at every denoising step.

In [ ]:
# All generation happens on the remote GPU via SlopClient.
# We only receive back numpy arrays.

all_latents: dict[str, list[np.ndarray]] = {}
all_images: dict[str, list[bytes]] = {}

with SlopClient(cfg) as client:
    info = client.get_server_info()
    print(f"Connected: {info.gpu_name} | CUDA {info.cuda_version}\n")

    for cat_name, prompt in CATEGORIES.items():
        print(f"\n{'='*60}")
        print(f"  {cat_name!r}  →  \"{prompt}\"")
        print(f"{'='*60}")

        results = client.generate_batch(
            prompt=prompt,
            n_variations=N_VARIATIONS,
            num_steps=NUM_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            model_id=MODEL,
            seed_start=SEED_START,
        )

        latents = [r.latents for r in results if r.latents is not None]
        images  = [r.image   for r in results if r.image is not None]
        all_latents[cat_name] = latents
        all_images[cat_name]  = images

        print(f"  ✓ {len(latents)} trajectories, shape {latents[0].shape}")

        # Save raw latents
        cat_dir = OUTPUT_DIR / cat_name
        cat_dir.mkdir(exist_ok=True)
        for i, lat in enumerate(latents):
            np.save(cat_dir / f"latents_{i:03d}.npy", lat)

print(f"\n✓ Generation complete — {sum(len(v) for v in all_latents.values())} total trajectories")

### Quick sanity check — show one sample image per category

In [ ]:
# Show 3 sample images per category (seeds 0, 1, 2)
n_show = min(3, N_VARIATIONS)
fig, axes = plt.subplots(len(CATEGORIES), n_show, figsize=(5 * n_show, 5 * len(CATEGORIES)))

for row, (cat, imgs) in enumerate(all_images.items()):
    for col in range(n_show):
        ax = axes[row, col] if len(CATEGORIES) > 1 else axes[col]
        if col < len(imgs):
            img = Image.open(BytesIO(imgs[col]))
            ax.imshow(img)
        ax.set_title(f"{cat} (seed={SEED_START + col})", fontsize=11)
        ax.axis("off")

plt.suptitle("Sample outputs per category", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_outputs.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Average Trajectories & Flow Fields

Averaging across seeds: individual noise patterns cancel, **systematic visual moves** survive.

In [ ]:
# All computation below is pure numpy — no GPU needed

avg_latents = {}
avg_flows = {}

for cat, stacks in all_latents.items():
    avg_latents[cat] = average_trajectories(stacks)
    avg_flows[cat]   = average_flow_field(stacks)
    print(f"{cat:15s}  avg latent {avg_latents[cat].shape}  |  avg flow {avg_flows[cat].shape}")

# Persist
for cat in avg_latents:
    np.save(OUTPUT_DIR / f"avg_latents_{cat}.npy", avg_latents[cat])
    np.save(OUTPUT_DIR / f"avg_flow_{cat}.npy", avg_flows[cat])
print("Saved.")

### Flow magnitude over time

How much does each category's latent *move* at each denoising step?

In [ ]:
colors = {"neutral": "#2196F3", "arab": "#FF5722", "palestinian": "#4CAF50"}

plt.figure(figsize=(12, 5))
for cat, stacks in all_latents.items():
    # Per-seed flow magnitude → mean ± SEM
    seed_mags = []
    for lat in stacks:
        flow = compute_flow_field(lat)
        mag = flow_magnitude(flow).mean(axis=(1, 2, 3))  # (steps,)
        seed_mags.append(mag)
    seed_mags = np.stack(seed_mags)  # (N, steps)
    mean_mag = seed_mags.mean(axis=0)
    sem_mag = seed_mags.std(axis=0, ddof=1) / np.sqrt(seed_mags.shape[0])

    c = colors.get(cat, "gray")
    plt.plot(mean_mag, label=f"{cat} (N={len(stacks)})", linewidth=2, color=c)
    plt.fill_between(
        range(len(mean_mag)),
        mean_mag - 1.96 * sem_mag,
        mean_mag + 1.96 * sem_mag,
        alpha=0.15, color=c,
    )

plt.xlabel("Denoising step")
plt.ylabel("Mean flow magnitude")
plt.title("Average latent velocity per category (mean ± 95% CI across seeds)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "flow_magnitude_over_time.png", dpi=150)
plt.show()

### Flow difference: category − neutral (batch-averaged)

Per-seed flow difference with 95% bootstrap CI. Gray spaghetti = individual seeds.

In [ ]:
non_baseline = [c for c in CATEGORIES if c != "neutral"]
batch_flow_results = {}

fig, axes_arr = plt.subplots(1, len(non_baseline), figsize=(12, 5))
if not hasattr(axes_arr, '__iter__'):
    axes_arr = [axes_arr]

for ax, cat in zip(axes_arr, non_baseline):
    bfr = batch_flow_difference(all_latents[cat], all_latents["neutral"])
    batch_flow_results[cat] = bfr
    c = colors.get(cat, "crimson")
    steps = np.arange(len(bfr.mean))

    # Per-seed curves (spaghetti)
    for curve in bfr.per_seed:
        ax.plot(curve, color="gray", alpha=0.15, linewidth=0.5)

    # Mean ± 95% CI
    ax.plot(steps, bfr.mean, linewidth=2.5, color=c, label="mean")
    ax.fill_between(steps, bfr.ci_lower, bfr.ci_upper, alpha=0.2, color=c, label="95% CI")
    ax.set_xlabel("Denoising step")
    ax.set_ylabel("|Δflow|")
    ax.set_title(f"{cat} − neutral  (N={bfr.per_seed.shape[0]} pairs)", fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Batch-averaged flow difference vs neutral baseline", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "flow_diff_vs_neutral.png", dpi=150)
plt.show()

---
## 3. Spatial Heatmaps — Mean, Std, and Signal-to-Noise Ratio

For each pixel in the 64×64 latent grid, compute the L2 channel-norm of the difference between categories — **for every seed pair**. Then report:
- **Mean**: where differences concentrate on average
- **Std**: where differences are noisy (vary across seeds)
- **SNR** (mean/std): where differences are *reliably* present. SNR > 2 ≈ robust signal.

In [ ]:
batch_spatial_results = {}

for cat in non_baseline:
    # Batch spatial at multiple timesteps
    n_total = all_latents[cat][0].shape[0]
    check_steps = np.linspace(0, n_total - 1, 4, dtype=int)

    fig, axes = plt.subplots(3, 4, figsize=(20, 14))

    for col, s in enumerate(check_steps):
        bsr = batch_spatial_heatmap(all_latents[cat], all_latents["neutral"], step=int(s))
        if col == len(check_steps) - 1:
            batch_spatial_results[cat] = bsr  # keep final-step result

        # Row 0: Mean heatmap
        im0 = axes[0, col].imshow(bsr.mean, cmap="inferno", interpolation="bilinear")
        axes[0, col].set_title(f"step {s} — mean", fontsize=10)
        axes[0, col].axis("off")
        fig.colorbar(im0, ax=axes[0, col], fraction=0.046)

        # Row 1: Std heatmap
        im1 = axes[1, col].imshow(bsr.std, cmap="viridis", interpolation="bilinear")
        axes[1, col].set_title(f"step {s} — std", fontsize=10)
        axes[1, col].axis("off")
        fig.colorbar(im1, ax=axes[1, col], fraction=0.046)

        # Row 2: SNR heatmap
        im2 = axes[2, col].imshow(bsr.snr, cmap="RdYlGn", interpolation="bilinear",
                                   vmin=0, vmax=max(3, np.percentile(bsr.snr, 95)))
        axes[2, col].set_title(f"step {s} — SNR", fontsize=10)
        axes[2, col].axis("off")
        fig.colorbar(im2, ax=axes[2, col], fraction=0.046)

    axes[0, 0].set_ylabel("Mean |diff|", fontsize=12)
    axes[1, 0].set_ylabel("Std across seeds", fontsize=12)
    axes[2, 0].set_ylabel("SNR = mean/std", fontsize=12)

    fig.suptitle(
        f"Spatial heatmaps: {cat} vs neutral  (N={min(len(all_latents[cat]), len(all_latents['neutral']))} seed pairs)\n"
        f"SNR > 2 = reliable signal (green). SNR < 1 = mostly noise (red).",
        fontsize=13,
    )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"spatial_heatmap_{cat}.png", dpi=150)
    plt.show()

---
## 4. Trajectory Divergence — Batch Statistics with Permutation Test

For every matched seed pair (i), compute the L2 distance between the two category trajectories at each step. This gives a **distribution** of divergence curves, not one curve.

- **Mean ± 95% bootstrap CI**: Is the divergence real or just seed noise?
- **Lock-in step**: mean ± std across seeds — when does the model commit?
- **Permutation test** (N=1000): Shuffle category labels, recompute. p < 0.05 means the divergence is statistically significant.

In [ ]:
batch_div_results = {}

fig, axes_arr = plt.subplots(1, len(non_baseline), figsize=(7 * len(non_baseline), 6))
if not hasattr(axes_arr, '__iter__'):
    axes_arr = [axes_arr]

for ax, cat in zip(axes_arr, non_baseline):
    print(f"Computing batch divergence: {cat} vs neutral ...")
    bdr = batch_divergence(
        all_latents[cat], all_latents["neutral"],
        n_perms=1000, ci=0.95,
    )
    batch_div_results[cat] = bdr
    c = colors.get(cat, "gray")
    steps = np.arange(len(bdr.mean))

    # Per-seed spaghetti
    for curve in bdr.per_seed:
        ax.plot(curve, color="gray", alpha=0.12, linewidth=0.5)

    # Mean ± 95% CI
    ax.plot(steps, bdr.mean, linewidth=2.5, color=c, label="mean")
    ax.fill_between(steps, bdr.ci_lower, bdr.ci_upper, alpha=0.2, color=c, label="95% CI")

    # Lock-in band
    if bdr.lock_in_mean >= 0:
        ax.axvspan(
            max(0, bdr.lock_in_mean - bdr.lock_in_std),
            bdr.lock_in_mean + bdr.lock_in_std,
            alpha=0.08, color=c,
        )
        ax.axvline(bdr.lock_in_mean, color=c, linestyle="--", alpha=0.6)
        ax.annotate(
            f"lock-in {bdr.lock_in_mean:.1f} ± {bdr.lock_in_std:.1f}",
            xy=(bdr.lock_in_mean, bdr.mean[int(bdr.lock_in_mean)]),
            xytext=(bdr.lock_in_mean + 3, bdr.mean[int(bdr.lock_in_mean)] * 1.1),
            fontsize=10, color=c,
            arrowprops=dict(arrowstyle="->", color=c, alpha=0.6),
        )

    # p-value annotation
    sig = "***" if bdr.p_value < 0.001 else "**" if bdr.p_value < 0.01 else "*" if bdr.p_value < 0.05 else "n.s."
    ax.text(
        0.98, 0.05,
        f"p = {bdr.p_value:.4f} {sig}\nN = {bdr.n_seeds_a}×{bdr.n_seeds_b}",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=11, bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"),
    )

    ax.set_xlabel("Denoising step")
    ax.set_ylabel("L2 distance from neutral")
    ax.set_title(f"{cat} vs neutral", fontsize=13)
    ax.legend(fontsize=9, loc="upper left")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Trajectory divergence — batch-averaged with permutation test\n"
    "Gray lines = individual seed pairs. Shaded band = lock-in step ± 1σ.",
    fontsize=14,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "trajectory_divergence.png", dpi=150)
plt.show()

# Print summary table
print(f"\n{'Category':<15} {'Final div':>12} {'± SEM':>10} {'Lock-in':>12} {'p-value':>10}")
print("─" * 62)
for cat, bdr in batch_div_results.items():
    print(
        f"{cat:<15} {bdr.mean[-1]:>12.4f} {bdr.sem[-1]:>10.4f} "
        f"{bdr.lock_in_mean:>8.1f} ± {bdr.lock_in_std:<3.1f} "
        f"{bdr.p_value:>10.4f}"
    )

In [ ]:
---
## 5. Per-Channel Analysis — Which Latent Channels Carry the Signal?

SD v1.5 has 4 latent channels. For each seed pair, compute the per-channel mean |difference|. Report mean ± SEM with error bars — if a channel's error bar doesn't include zero, it reliably carries category-specific information.

In [ ]:
batch_ch_results = {}

fig, axes_arr = plt.subplots(1, len(non_baseline), figsize=(7 * len(non_baseline), 5))
if not hasattr(axes_arr, '__iter__'):
    axes_arr = [axes_arr]

for ax, cat in zip(axes_arr, non_baseline):
    bcr = batch_channel_difference(all_latents[cat], all_latents["neutral"], step=-1)
    batch_ch_results[cat] = bcr

    x = np.arange(len(bcr.mean))
    c = colors.get(cat, "gray")

    # Bar plot with error bars (SEM)
    ax.bar(x, bcr.mean, yerr=1.96 * bcr.sem, capsize=6,
           color=c, alpha=0.7, edgecolor="black", linewidth=0.5)

    # Individual seed dots
    for seed_vals in bcr.per_seed:
        ax.scatter(x + np.random.uniform(-0.15, 0.15, len(x)), seed_vals,
                   color="gray", alpha=0.3, s=15, zorder=5)

    ax.set_xlabel("Latent channel")
    ax.set_ylabel("Mean |difference|")
    ax.set_title(f"{cat} vs neutral (final step, N={bcr.per_seed.shape[0]})", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels([f"ch{i}" for i in x])
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(
    "Per-channel difference: bars = mean, whiskers = 95% CI, dots = individual seeds",
    fontsize=13,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_channel_diff.png", dpi=150)
plt.show()

# Also show across timesteps
fig2, axes2 = plt.subplots(1, len(non_baseline), figsize=(7 * len(non_baseline), 5))
if not hasattr(axes2, '__iter__'):
    axes2 = [axes2]

for ax, cat in zip(axes2, non_baseline):
    n_total = all_latents[cat][0].shape[0]
    check_steps = np.linspace(0, n_total - 1, 6, dtype=int)

    for s in check_steps:
        bcr_s = batch_channel_difference(all_latents[cat], all_latents["neutral"], step=int(s))
        ax.errorbar(
            np.arange(len(bcr_s.mean)) + int(s) * 0.05,
            bcr_s.mean, yerr=bcr_s.sem,
            fmt="o-", capsize=3, alpha=0.7, label=f"step {s}",
        )

    ax.set_xlabel("Latent channel")
    ax.set_ylabel("Mean |difference| ± SEM")
    ax.set_title(f"{cat} vs neutral — temporal evolution", fontsize=12)
    ax.set_xticks(range(4))
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Channel-level difference across denoising steps", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_channel_temporal.png", dpi=150)
plt.show()

---
## 6. Inter-Category PCA — Fitted on All Seeds

PCA is fitted on sampled timesteps from **every individual seed** (not just averages). This prevents the PCA axes from overfitting to the mean trajectory. Per-seed projections are shown as transparent spaghetti; the mean as a thick line.

In [ ]:
avg_proj, per_seed_proj, ic_pca, ic_var = batch_inter_category_pca(all_latents)

print("Inter-category PCA — explained variance (fitted on ALL seeds):")
for i, ev in enumerate(ic_var):
    bar = "█" * int(ev * 200)
    print(f"  PC{i}: {ev:.4f}  {bar}")
print(f"  Total: {ic_var.sum():.4f}")
print(f"  Training points: {sum(len(s) for s in all_latents.values())} seeds × sampled steps")

# Plot mean ± per-seed spread for each PC
n_pcs = min(3, len(ic_var))
fig, axes_arr = plt.subplots(1, n_pcs, figsize=(6 * n_pcs, 5))
if n_pcs == 1:
    axes_arr = [axes_arr]

for pc_idx, ax in enumerate(axes_arr):
    for cat in CATEGORIES:
        c = colors.get(cat, "gray")
        # Per-seed spaghetti
        seed_projs = per_seed_proj[cat]  # (N, T, n_comp)
        for sp in seed_projs:
            ax.plot(sp[:, pc_idx], color=c, alpha=0.1, linewidth=0.5)
        # Mean
        ax.plot(avg_proj[cat][:, pc_idx], label=cat, linewidth=2.5, color=c)

    ax.set_title(f"PC{pc_idx} ({ic_var[pc_idx]:.1%} var)", fontsize=13)
    ax.set_xlabel("Denoising step")
    ax.set_ylabel("Projection")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Per-seed trajectory projections onto inter-category PC axes\n"
    "Thin lines = individual seeds, thick = mean. Separation = systematic.",
    fontsize=14,
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "inter_category_pca.png", dpi=150)
plt.show()

### 2D trajectory plot — PC0 vs PC1, all seeds

Each seed's trajectory in PC space, with mean trajectory highlighted. Clear category clusters = systematic bias; overlapping clouds = noise.

In [ ]:
plt.figure(figsize=(10, 8))

for cat in CATEGORIES:
    c = colors.get(cat, "gray")

    # Per-seed trajectories (thin)
    for sp in per_seed_proj[cat]:
        plt.plot(sp[:, 0], sp[:, 1], color=c, alpha=0.08, linewidth=0.5)

    # Mean trajectory (thick)
    proj = avg_proj[cat]
    plt.plot(proj[:, 0], proj[:, 1], linewidth=2.5, color=c, alpha=0.9)
    plt.scatter(proj[0, 0], proj[0, 1], s=100, color=c, marker="o",
                zorder=5, edgecolors="black")
    plt.scatter(proj[-1, 0], proj[-1, 1], s=150, color=c, marker="*",
                zorder=5, edgecolors="black", label=cat)

    # Per-seed final points (cloud)
    finals = per_seed_proj[cat][:, -1, :2]  # (N, 2)
    plt.scatter(finals[:, 0], finals[:, 1], s=30, color=c, alpha=0.4, zorder=4)

plt.xlabel(f"PC0 ({ic_var[0]:.1%} var)")
plt.ylabel(f"PC1 ({ic_var[1]:.1%} var)")
plt.title(
    "Denoising trajectories in PC space (all seeds)\n"
    "○ = start, ★ = mean final, dots = per-seed finals"
)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "trajectory_2d.png", dpi=150)
plt.show()

---
## 7. Summary Statistics Table

All batch-averaged metrics in one place: divergence, lock-in step, per-channel signal, p-values.

In [ ]:
# ── Summary table ──
summary = {}

print(f"{'Category':<14} {'Final div':>11} {'± SEM':>8} {'Lock-in':>14} {'p-value':>9} {'Top ch':>8} {'Spatial SNR':>12}")
print("─" * 80)

for cat in non_baseline:
    bdr = batch_div_results[cat]
    bcr = batch_ch_results[cat]
    bsr = batch_spatial_results[cat]

    top_ch = int(np.argmax(bcr.mean))
    median_snr = float(np.median(bsr.snr))

    summary[cat] = {
        "final_divergence": float(bdr.mean[-1]),
        "final_divergence_sem": float(bdr.sem[-1]),
        "lock_in_mean": float(bdr.lock_in_mean),
        "lock_in_std": float(bdr.lock_in_std),
        "p_value": float(bdr.p_value),
        "top_channel": top_ch,
        "top_channel_diff": float(bcr.mean[top_ch]),
        "top_channel_sem": float(bcr.sem[top_ch]),
        "median_spatial_snr": median_snr,
        "n_seeds": bdr.n_seeds_a,
    }

    sig = "***" if bdr.p_value < 0.001 else "**" if bdr.p_value < 0.01 else "*" if bdr.p_value < 0.05 else "n.s."
    print(
        f"{cat:<14} {bdr.mean[-1]:>11.4f} {bdr.sem[-1]:>8.4f} "
        f"{bdr.lock_in_mean:>7.1f} ± {bdr.lock_in_std:<4.1f} "
        f"{bdr.p_value:>8.4f} {sig} "
        f"{'ch' + str(top_ch):>5} "
        f"{median_snr:>12.2f}"
    )

# Save
with open(OUTPUT_DIR / "summary_metrics.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nSaved to {OUTPUT_DIR / 'summary_metrics.json'}")

---
## 7a. Divergence Rate Analysis — *How Fast* Does Each Category Separate?

Absolute divergence is uninformative for related prompts: "Arab in Jaffa" and "Jesus in Jaffa" will both diverge from neutral. The question is the **rate profile** $\frac{d}{dt}\|L_A(t) - L_\text{neutral}(t)\|_2$:

- Which category starts diverging **earliest**? (peak rate step)
- Which accumulates the most separation per step? (rate magnitude)
- Rate ratio between categories: does "Arab" accelerate before "Palestinian"?

All computed per-seed, then mean ± CI.

---
## 8. Composite Figure — One Panel Per Category

For each non-neutral category, a single publication-quality figure combining:
- Row 1: Spatial SNR heatmaps at key timesteps
- Row 2: Flow difference (batch) + divergence (batch with CI)
- Row 3: PCA projections (per-seed + mean)

In [ ]:
for cat in non_baseline:
    bdr = batch_div_results[cat]
    bfr = batch_flow_results[cat]
    c = colors.get(cat, "gray")

    fig = plt.figure(figsize=(22, 16))
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.35, wspace=0.3)

    # ── Row 1: Spatial SNR heatmaps at 4 timesteps ──
    n_total = all_latents[cat][0].shape[0]
    check_steps = np.linspace(0, n_total - 1, 4, dtype=int)
    step_labels = [f"t={s}" for s in check_steps]

    for col, (s, label) in enumerate(zip(check_steps, step_labels)):
        ax = fig.add_subplot(gs[0, col])
        bsr = batch_spatial_heatmap(all_latents[cat], all_latents["neutral"], step=int(s))
        im = ax.imshow(bsr.snr, cmap="RdYlGn", interpolation="bilinear",
                       vmin=0, vmax=max(3, np.percentile(bsr.snr, 95)))
        ax.set_title(f"{label}\nmed SNR={np.median(bsr.snr):.1f}", fontsize=10)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)

    # ── Row 2 left: Flow difference ──
    ax_flow = fig.add_subplot(gs[1, :2])
    steps = np.arange(len(bfr.mean))
    for curve in bfr.per_seed[:5]:  # show max 5 seeds
        ax_flow.plot(curve, color="gray", alpha=0.2, linewidth=0.5)
    ax_flow.plot(steps, bfr.mean, linewidth=2.5, color=c)
    ax_flow.fill_between(steps, bfr.ci_lower, bfr.ci_upper, alpha=0.2, color=c)
    ax_flow.set_xlabel("Denoising step")
    ax_flow.set_ylabel("|Δflow| ± CI")
    ax_flow.set_title(f"Flow difference: {cat} − neutral", fontsize=12)
    ax_flow.grid(True, alpha=0.3)

    # ── Row 2 right: Divergence curve ──
    ax_div = fig.add_subplot(gs[1, 2:])
    steps_d = np.arange(len(bdr.mean))
    for curve in bdr.per_seed[:5]:
        ax_div.plot(curve, color="gray", alpha=0.2, linewidth=0.5)
    ax_div.plot(steps_d, bdr.mean, linewidth=2.5, color=c)
    ax_div.fill_between(steps_d, bdr.ci_lower, bdr.ci_upper, alpha=0.2, color=c)
    if bdr.lock_in_mean >= 0:
        ax_div.axvline(bdr.lock_in_mean, color="red", linestyle="--", alpha=0.7)
    sig = "***" if bdr.p_value < 0.001 else "**" if bdr.p_value < 0.01 else "*" if bdr.p_value < 0.05 else "n.s."
    ax_div.text(0.97, 0.05, f"p={bdr.p_value:.4f} {sig}",
                transform=ax_div.transAxes, ha="right", fontsize=11,
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"))
    ax_div.set_xlabel("Denoising step")
    ax_div.set_ylabel("L2 divergence from neutral")
    ax_div.set_title("Trajectory divergence (with permutation test)", fontsize=12)
    ax_div.grid(True, alpha=0.3)

    # ── Row 3: PCA projections (top 4 PCs) ──
    n_show_pcs = min(4, len(ic_var))
    for pc_idx in range(n_show_pcs):
        ax_pc = fig.add_subplot(gs[2, pc_idx])
        ev = ic_var[pc_idx]
        # Per-seed spaghetti for this category and neutral
        for sp in per_seed_proj["neutral"][:3]:
            ax_pc.plot(sp[:, pc_idx], color=colors["neutral"], alpha=0.15, linewidth=0.5)
        for sp in per_seed_proj[cat][:3]:
            ax_pc.plot(sp[:, pc_idx], color=c, alpha=0.15, linewidth=0.5)
        # Means
        ax_pc.plot(avg_proj["neutral"][:, pc_idx], linewidth=2,
                   color=colors["neutral"], label="neutral")
        ax_pc.plot(avg_proj[cat][:, pc_idx], linewidth=2, color=c, label=cat)
        ax_pc.set_title(f"PC{pc_idx} ({ev:.1%})", fontsize=11)
        ax_pc.set_xlabel("Step")
        ax_pc.legend(fontsize=8)
        ax_pc.grid(True, alpha=0.3)

    fig.suptitle(
        f"Colonial Visual Grammar: \"{CATEGORIES[cat]}\" vs. neutral\n"
        f"N={N_VARIATIONS} seeds/category · {NUM_STEPS} steps · "
        f"p={bdr.p_value:.4f} · lock-in step {bdr.lock_in_mean:.0f} ± {bdr.lock_in_std:.0f}",
        fontsize=15, y=1.01,
    )
    plt.savefig(OUTPUT_DIR / f"composite_{cat}.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"\n✓ All results saved to {OUTPUT_DIR}")

---
## 9. Effect Size & Power Analysis

How many seeds do we need for reliable detection? Compute Cohen's d from current data and estimate required N for 80% power.

In [ ]:
# Effect size: Cohen's d = (mean_divergence - 0) / pooled_std
# Since the null hypothesis is zero divergence, d = mean / std

print(f"{'Category':<14} {'Cohen d':>9} {'Needed N (80% power)':>22}")
print("─" * 50)

for cat in non_baseline:
    bdr = batch_div_results[cat]
    final_divs = bdr.per_seed[:, -1]  # per-seed final divergence values

    d = final_divs.mean() / (final_divs.std(ddof=1) + 1e-10)

    # Approximate required N for paired t-test at α=0.05, power=0.80
    # N ≈ (z_α + z_β)² / d²  where z_0.025=1.96, z_0.20=0.84
    required_n = ((1.96 + 0.84) ** 2) / (d ** 2 + 1e-10)
    required_n = max(2, int(np.ceil(required_n)))

    print(f"{cat:<14} {d:>9.2f} {required_n:>22d}")

    # Accumulation plot: how does p-value change with N?
    if bdr.per_seed.shape[0] >= 4:
        ns = range(2, bdr.per_seed.shape[0] + 1)
        p_by_n = []
        for n_sub in ns:
            sub_divs = bdr.per_seed[:n_sub, -1]
            # One-sample t-test against 0
            from scipy import stats
            t_stat, p_val = stats.ttest_1samp(sub_divs, 0)
            p_by_n.append(p_val)

        plt.figure(figsize=(8, 4))
        plt.plot(list(ns), p_by_n, "o-", color=colors.get(cat, "gray"), linewidth=2)
        plt.axhline(0.05, color="red", linestyle="--", alpha=0.5, label="α = 0.05")
        plt.xlabel("Number of seeds used")
        plt.ylabel("p-value (t-test)")
        plt.title(f"Power accumulation: {cat} — how many seeds for significance?")
        plt.yscale("log")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"power_accumulation_{cat}.png", dpi=150)
        plt.show()

---
## Methodology Notes

### Statistical Design

Every metric is computed **per-seed first**, then aggregated:

| Metric | Per-seed computation | Aggregation |
|--------|---------------------|-------------|
| Trajectory divergence | L2(lat_a[i], lat_b[i]) at each step | Mean ± 95% bootstrap CI |
| Flow difference | \|flow_a[i] - flow_b[i]\| | Mean ± 95% bootstrap CI |
| Channel signal | per-channel \|diff\| | Mean ± SEM, individual dots |
| Spatial heatmap | per-pixel L2 channel-norm | Mean, Std, SNR = mean/std |
| PCA projection | Full trajectory → PC space | Per-seed spaghetti + mean |
| Lock-in step | 50%-of-final threshold | Mean ± std across seeds |

**Permutation test** (N=1000): Pool all trajectories from both categories, randomly reassign labels, recompute mean final divergence. p-value = fraction of permutations ≥ observed.

**Effect size**: Cohen's d computed from per-seed final divergence values. Power analysis estimates N needed for α=0.05, 80% power.

### Colonial Archive (Optional Extension)

If colonial archival images are available (see `dataset.md`), they can be VAE-encoded and used to extract "colonial PC axes" via `set_colonial_axes()`. This enables projecting diffusion trajectories onto axes defined by historical colonial visual grammar — a separate, more specific analysis.

### Prompts

Prompt pairs are chosen to isolate the **category variable** while keeping context constant:
- "a person in daily life" vs "an Arab person in daily life"
- Same syntax, same scene — only the identity marker differs
- Any systematic latent-space divergence is attributable to the model's learned associations